# 06 - Document Intelligence

**AI-900 Domain:** Describe features of computer vision workloads on Azure

## What You'll Learn
- How **Azure AI Document Intelligence** extracts structured data from documents
- Analyze **invoices** and extract fields like vendor name, total amount, and line items
- Analyze **receipts** and extract purchase information
- Use **pre-built models** that work out of the box

> **Just click `Run All` - no coding needed!**

## Setup: Load Credentials

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(os.path.join("..", ".env"))

DOC_ENDPOINT = os.environ.get("AZURE_DOC_INTEL_ENDPOINT", "").rstrip("/")
DOC_KEY = os.environ.get("AZURE_DOC_INTEL_KEY", "")

# Fall back to the multi-service key if doc-specific key isn't set
if not DOC_KEY:
    DOC_ENDPOINT = os.environ.get("AZURE_AI_ENDPOINT", "").rstrip("/")
    DOC_KEY = os.environ.get("AZURE_AI_KEY", "")

if not DOC_KEY:
    raise ValueError("Neither AZURE_DOC_INTEL_KEY nor AZURE_AI_KEY is set! Please check your .env file.")

print(f"Credentials loaded!")
print(f"Endpoint: {DOC_ENDPOINT[:50]}...")

---
## Part 1: Analyze an Invoice

Azure AI Document Intelligence has **pre-built models** for common document types:
- Invoices
- Receipts
- Business cards
- ID documents
- Tax forms (W2, 1099)

Let's analyze a sample invoice using the pre-built invoice model.

In [ ]:
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest
from azure.core.credentials import AzureKeyCredential

# Create the Document Intelligence client
doc_client = DocumentIntelligenceClient(
    endpoint=DOC_ENDPOINT,
    credential=AzureKeyCredential(DOC_KEY)
)

print("Document Intelligence client created!")

In [ ]:
# Use Microsoft's sample invoice URL
invoice_url = "https://raw.githubusercontent.com/Azure-Samples/cognitive-services-REST-api-samples/master/curl/form-recognizer/sample-invoice.pdf"

print("Analyzing sample invoice...")
print(f"URL: {invoice_url}\n")

# Analyze the invoice
poller = doc_client.begin_analyze_document(
    "prebuilt-invoice",
    AnalyzeDocumentRequest(url_source=invoice_url)
)
result = poller.result()

print("Analysis complete!")
print(f"Documents found: {len(result.documents)}")

In [ ]:
# Display extracted invoice fields
print("=" * 60)
print("INVOICE ANALYSIS RESULTS")
print("=" * 60)

for doc in result.documents:
    fields = doc.fields
    
    # Helper to safely get field values
    def get_field(name):
        field = fields.get(name)
        if field:
            content = field.content or field.value_string or str(field.value_number) if hasattr(field, 'value_number') and field.value_number else field.content
            confidence = field.confidence if field.confidence else 0
            return content, confidence
        return "Not found", 0
    
    # Display key fields
    key_fields = [
        ("VendorName", "Vendor"),
        ("VendorAddress", "Vendor Address"),
        ("CustomerName", "Customer"),
        ("InvoiceId", "Invoice ID"),
        ("InvoiceDate", "Invoice Date"),
        ("DueDate", "Due Date"),
        ("SubTotal", "Subtotal"),
        ("TotalTax", "Tax"),
        ("InvoiceTotal", "Total"),
    ]
    
    print(f"\n{'Field':<20} {'Value':<35} {'Confidence':<10}")
    print("-" * 65)
    
    for field_name, display_name in key_fields:
        field = fields.get(field_name)
        if field:
            value = field.content or "N/A"
            conf = f"{field.confidence:.1%}" if field.confidence else "N/A"
            print(f"{display_name:<20} {str(value):<35} {conf:<10}")

### Line Items

Document Intelligence also extracts **individual line items** from invoices.

In [ ]:
for doc in result.documents:
    items_field = doc.fields.get("Items")
    if items_field and items_field.value_array:
        print("\nLINE ITEMS")
        print("=" * 60)
        print(f"{'Description':<30} {'Qty':<6} {'Unit Price':<12} {'Amount':<12}")
        print("-" * 60)
        
        for item in items_field.value_array:
            if item.value_object:
                desc = item.value_object.get("Description")
                qty = item.value_object.get("Quantity")
                price = item.value_object.get("UnitPrice")
                amount = item.value_object.get("Amount")
                
                desc_val = desc.content if desc else "N/A"
                qty_val = qty.content if qty else "N/A"
                price_val = price.content if price else "N/A"
                amount_val = amount.content if amount else "N/A"
                
                print(f"{str(desc_val)[:28]:<30} {str(qty_val):<6} {str(price_val):<12} {str(amount_val):<12}")
    else:
        print("\nNo line items found.")

---
## Part 2: Analyze a Receipt

The receipt model extracts purchase information from store receipts.

In [ ]:
# Use Microsoft's sample receipt URL
receipt_url = "https://raw.githubusercontent.com/Azure-Samples/cognitive-services-REST-api-samples/master/curl/form-recognizer/contoso-receipt.png"

print("Analyzing sample receipt...\n")

poller = doc_client.begin_analyze_document(
    "prebuilt-receipt",
    AnalyzeDocumentRequest(url_source=receipt_url)
)
receipt_result = poller.result()

print("=" * 60)
print("RECEIPT ANALYSIS RESULTS")
print("=" * 60)

for doc in receipt_result.documents:
    fields = doc.fields
    
    receipt_fields = [
        ("MerchantName", "Store Name"),
        ("MerchantAddress", "Store Address"),
        ("MerchantPhoneNumber", "Phone"),
        ("TransactionDate", "Date"),
        ("TransactionTime", "Time"),
        ("Subtotal", "Subtotal"),
        ("TotalTax", "Tax"),
        ("Total", "Total"),
    ]
    
    print(f"\n{'Field':<20} {'Value':<40}")
    print("-" * 60)
    
    for field_name, display_name in receipt_fields:
        field = fields.get(field_name)
        if field:
            value = field.content or field.value_string or "N/A"
            print(f"{display_name:<20} {str(value):<40}")
    
    # Display items
    items = fields.get("Items")
    if items and items.value_array:
        print(f"\nItems Purchased:")
        print("-" * 40)
        for item in items.value_array:
            if item.value_object:
                desc = item.value_object.get("Description")
                total = item.value_object.get("TotalPrice")
                if desc:
                    total_str = f" - {total.content}" if total else ""
                    print(f"  {desc.content}{total_str}")

---
## Part 3: General Document Analysis (Layout)

The **Layout** model extracts text, tables, and structure from any document.

In [ ]:
print("Analyzing document layout...\n")

poller = doc_client.begin_analyze_document(
    "prebuilt-layout",
    AnalyzeDocumentRequest(url_source=invoice_url)
)
layout_result = poller.result()

print("=" * 60)
print("LAYOUT ANALYSIS RESULTS")
print("=" * 60)

# Show pages info
print(f"\nPages: {len(layout_result.pages)}")
for page in layout_result.pages:
    print(f"  Page {page.page_number}: {page.width} x {page.height} ({page.unit})")
    if page.lines:
        print(f"  Lines of text: {len(page.lines)}")

# Show tables
if layout_result.tables:
    print(f"\nTables found: {len(layout_result.tables)}")
    for i, table in enumerate(layout_result.tables):
        print(f"  Table {i+1}: {table.row_count} rows x {table.column_count} columns")
else:
    print("\nNo tables found.")

# Show first few lines of extracted text
if layout_result.pages and layout_result.pages[0].lines:
    print("\nFirst 10 lines of extracted text:")
    print("-" * 40)
    for line in layout_result.pages[0].lines[:10]:
        print(f"  {line.content}")

---
## Key Concepts for AI-900

| Concept | Description |
|---------|-------------|
| **Document Intelligence** | Azure service for extracting structured data from documents |
| **Pre-built Models** | Ready-to-use models for invoices, receipts, IDs, etc. |
| **Custom Models** | You can train models for your specific document types |
| **Layout Analysis** | Extracts text, tables, and structure from any document |
| **Confidence Score** | How sure the model is about each extracted field |
| **Fields** | Named data points extracted from documents (e.g., VendorName, Total) |

### AI-900 Exam Tips
- Azure AI Document Intelligence was previously called **Form Recognizer**
- **Pre-built models** require no training and work immediately
- **Custom models** need training data (labeled example documents)
- The **Layout** model works on any document type
- The service can process **PDFs, images (JPEG, PNG, TIFF), and Office files**
- Each extracted field includes a **confidence score**

---
## Congratulations!

You've completed all the AI-900 hands-on notebooks! Here's what you covered:

| Notebook | AI-900 Domain |
|----------|---------------|
| 01 - Responsible AI | AI Principles & Content Safety |
| 02 - Computer Vision | Image Analysis & OCR |
| 03 - Natural Language | Sentiment, NER, Translation |
| 04 - Speech AI | Text-to-Speech & Speech-to-Text |
| 05 - Generative AI | GPT Models & Prompt Engineering |
| 06 - Document Intelligence | Document Parsing & Extraction |

**Good luck on the AI-900 exam!**